# Classification and Loss Function

## Classification Models

### Classification and Regression
- Regression: 
    - The target (variable of outcome) of prediction is continous (e.g. house price, earnings, etc.)
    - The model predicts a continous distribution over the output $y$ for each set of covariates $x$
- Classification: 
    - The target (variable of outcome) of prediction is discrete (e.g. product choice, default vs no default, employment status)
    - The model predicts a discrete probability distribution (histogram) over the output $y$ for each set of covariates $x$

### Discrete Choice Model
- Set Up:
    - Outcome (target): $y_i \in {1,\ldots,J}$ (choice among $J$ alternatives)
    - Covariates (features): $\mathbf{X}_i$
- Goal of the Model: 
    - Predicting $P(y_i = j | \mathbf{X}_i)$ for given $\mathbf{X}_i$
    - The label $j$ with highest predicted probability of relaization is the prediction made by the model

### Multinomial Logit Model (Example of Discrete Choise Model)
- Foundation (Random Utility Theory):
    - Assume that an agent chooses alternative $j$ with highest utility
    - The utility of an alternative is $U_{ij} = \mathbf{X}_i \beta_j + \varepsilon_{ij}$
    - The agent chooses $j$ iff $U_{ij} > U_{ia}$ for all $a \neq j$
- Multinomial Logit Model:
    - If we further assume that $\varepsilon_{ij}$ follows an i.i.d. Gumbel distribution (extreme value type I)
    - Then choice probabilities for any alternative $j$ have a closed form (multinomial logit, or called **soft max**)
    - That is: $$P(y_i = j | \mathbf{X}_i) = \frac{\exp(\mathbf{X}_i \beta_j)}{\sum_{a=1}^J \exp(\mathbf{X}_i \beta_a)}$$

## Loss Functions

### What is a Good Model
- When we train machine learning models, we seek the parameters that produce the best possible mapping from input to output
- Consider the model as computing a conditional probability distribution $Pr(y|x)$ over possible outputs $y$ given input $x$
- The "best" possible mapping should minimize the deviance of the model's predicted conditional distribution to the empirical conditional distribution
- This implies that we can use a measure of such deviance as the loss function that our model need to minimize during the training process

### Cross Entropy Loss
- Foundation:
    - The cross entropy loss is based on the idea of minimzing the deviance of a model's predicted conditional distribution to the empirical conditional distribution
    - Remember that the Kullback-Leibler (KL) divergence measures deviance between two probability distributions
    - Then the cross entropy loss aims to minimize the KL deviance from model distribution $Q_{\theta}(y|x)$ to empirical distribution $P(y|x)$
    - Which is: 
    $$ 
    \begin{align*}
    \hat{\theta} & = \arg\min_{\theta} D_{KL}[P||Q] \\
    & = \arg\min_{\theta} \int_{-\infty}^{\infty} p(y|x) \log \frac{p(y|x)}{q_{\theta}(y|x)} dy \\
    & = \arg\min_{\theta} \int_{-\infty}^{\infty} p(y|x) \log p(y|x) dy  - \int_{-\infty}^{\infty} p(y|x) \log q_{\theta}(y|x) dy  \\
    \end{align*}
    $$
    - As the first part does not depend on $\theta$:
    $$
    \hat{\theta} = \arg\min_{\theta} - \int_{-\infty}^{\infty} p(y|x) \log q_{\theta}(y|x) dy 
    $$
    - The remaining term that \theat need to minimize is called **cross-entrophy**
    - It is the amount of uncertainty that remains in emprical distribution after taking into account what we already know from the predicted distribution
- Defining Empirical Distribution:
    - Consider that we observe a set of empirical data ${\{y_i\}}^I_{i=1}$ at $x$
    - We can describe the empirical conditional distribution as a weighted sum of point masses $$ p(y|x) = \frac{1}{I} \sum_{i=1}^I\delta[y-y_i] $$
    - Where $\delta(z)$ is the Dirac delta function:
        - It equals 0 if $z \neq 0$
        - It equals $\infty$ if $z = 0$
        - $\int_{-\infty}^{\infty} \delta(z) dz = 1$
    - Insert the representation of empirical distribution, we get: 
    $$
    \begin{align*}
    \hat{\theta} & = \arg\min_{\theta} - \int_{-\infty}^{\infty} \left(\frac{1}{I}\sum_{i=1}^I\delta[y-y_i]\right) \log q_{\theta}(y|x) dy \\
    & = \arg\min_{\theta} - \frac{1}{I}\sum_{i=1}^I \left( \int_{-\infty}^{\infty} \delta[y-y_i] \log q_{\theta}(y|x) dy \right) \\
    & = \arg\min_{\theta} - \frac{1}{I} \sum_{i=1}^I \log q_{\theta}(y_i|x_i) \\
    & = \arg\min_{\theta} - \sum_{i=1}^I \log q_{\theta}(y_i|x_i) 
    \end{align*}
    $$
- We can now see that minimizing cross-entropy (minimizing the deviance between the model and empirical data distributions) is equivalent to miniziming negative log-likelihood (maximizing the data likelihood)
- That is why **cross-entropy loss** and **negative log-likelihood loss** is equivalent and usually used interchangeably
- The exact form of the **cross-entropy loss** depends on the distribution $q_{\theta}(y_i|x_i)$ parametrized by ML model

### Minimizing Least Squares Loss
- Assuming that $y_i|x_i\sim N(f[x_i,\phi],\sigma^2)$, where $f[x_i,\phi]$ is the model's prediction (here the model only predict the mean of condititonal distribution)
- Then the model distribution is $$q_{\phi}(y_i|x_i) = \left( \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left( -\frac{(y_i-f[x_i,\phi])^2}{2\sigma^2} \right) \right)$$
- The parameter $\phi$ that minimizes the corss-entropy loss is:
$$
\begin{align*} 
\hat{\phi} &= \arg\min_{\phi} -\sum_{i=1}^I \log q(y_i|x_i) \\ 
&= \arg\min_{\phi} -\sum_{i=1}^I \log \left( \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left( -\frac{(y_i-f[x_i,\phi])^2}{2\sigma^2} \right) \right) \\
&= \arg\min_{\phi} \sum_{i=1}^I -\log \left( \frac{1}{\sqrt{2\pi\sigma^2}}\right) +\frac{(y_i-f[x_i,\phi])^2}{2\sigma^2} \\ 
&= \arg\min_{\phi} \sum_{i=1}^I (y_i-f[x_i,\phi])^2 
\end{align*}
$$
- Therefore, the parameter $\phi$ that minimizes the corss-entropy loss is also the parameter that minimizes the least squares loss 
- This justify the use of square loss functions (MSE): 
    - Minimizing MSE is equivalent to minimizing the deviance of model distribution from empirical distribution
    - Assuming that the prediction should follows a normal distribution characterized with the mean predicted by the model
- In other word, MSE is suitable if:
    - outcome are indepedent
    - drawn from a normal distribution with mean that can be predicted and with constant variance

### Heteroscedastic Regression
- In square loss function case, we assume that the model only predicts the mean of the model distribution (which is a normal distribution)
- However, there is nothing to stop us from treating the variance of the model distribution as another target of prediction
- Then the model solves the corss-entropy minimization problem:
$$
\begin{align*} 
\arg\min_{\phi} -\sum_{i=1}^I \log \left( \frac{1}{\sqrt{2\pi f_2[x_i,\phi]^2}}\exp\left( -\frac{(y_i-f_1[x_i,\phi])^2}{2 f_2[x_i,\phi]^2} \right) \right)
\end{align*}
$$
- where $f_1$ predicts the mean of the normal distribution that the prediction follows
- $f_2$ predicts the variance of the normal distribution that the prediction follows
- this is called heteroscedastic regression (as variance is heteroscedastic and depends on $x_i$)

### Binary Cross Entropy Loss
- Binary Classification:
    - The goal is to assign the data $x$ to one of two discrete classes $y ∈ {0, 1}$ (we refer to y as a label)
    - To construct cross-entrophy loss function we choose a probability distribution over the output space $y$
    - A suitable choice is the Bernoulli distribution: $$ y_i|x_i\sim Ber(\lambda_i) $$
    - That is:
        - $ Pr(y_i = 1|x_i) = \lambda_i $
        - $ Pr(y_i = 0|x_i) = 1- \lambda_i $
        - $ Pr(y_i|x_i) = {y_i} ^ {\lambda_i} (1-y_i)^ {(1-\lambda_i)} $
- Logistic Sigmoid:
    - We need to set the machine learning model $f[x, ϕ]$ to predict the single distribution parameter $λ$
    - However, $λ$ can only take values in the range $[0, 1]$, and we cannot guarantee that the network output will lie in this range
    - Consequently, we pass the prediction through a function that maps the real numbers $R$ to $[0, 1]$
    - A suitable function is the logistic sigmoid: $$logit[z] = \frac{1}{1 + exp (z)}$$
    - Therefore, the model predicts the single distribution parameter $λ$ as $$ λ_i = logit[f[x_i,\phi]]$$
- Cross Entrophy Loss:
    - The optimal parameter that minimizes cross entropy loss would be: 
    $$
    \begin{align*} 
    \hat{\phi} &= \arg\min_{\phi} -\sum_{i=1}^I \log \left( (1-logit(f[x_i,\phi]))^{1-y_i} logit(f[x_i,\phi])^{y_i} \right) \\ 
    &= \arg\min_{\phi} \sum_{i=1}^I -(1-y_i)\log \left(1-logit(f[x_i,\phi])\right) - y_i \log\left(logit(f[x_i,\phi]) \right) 
    \end{align*}
    $$


### Multiclass Cross Entropy Loss
- Multiclass Classification:
    - The goal is to assign an input data example $x$ to one of $K > 2$ classes, so $y ∈ {1, 2, . . . , K}$
    - To construct cross-entrophy loss function we again choose a probability distribution over the output space $y$
    - We can choose categorical distribution:
        - It has K parameters $λ_1, λ_2, . . . , λ_K$ 
        - It determines the probability of each category as $Pr(y = k) = λ_k$
- Softmax:
    - The parameters of categorical distribution are constrained to take values between zero and one, and they must collectively sum to one
    - If we use $f[x, ϕ]$ with $K$ outputs to directly predict these parameters, the outputs will not necessarily obey the constraints
    - Therefore, we pass the $K$ outputs through a function that ensures these constraints are respected
    - A suitable choice is the softmax function
    - This takes an arbitrary vector of length K and returns a same length vector with k-th element: 
    $$ softmax_k[z] = \frac{exp[z_k]}{\sum_{k'=1}^{K} exp[z_{k'}]}$$
    - Therefore, the model predicts any distribution parameter $λ_k$ as $$ λ_k = softmax_k[f[x_i,\phi]]$$
    - The conditional probability that input $x$ has label $y = k$ is $$Pr(y_i=k|x_i) = softmax_k[f[x_i,\phi]] $$
- Cross Entrophy Loss:
    - The optimal parameter that minimizes cross entropy loss would be: 
    $$
    \hat{\phi} = \arg\min_{\phi} -\sum_{i=1}^I \log \left( softmax_{y_i}(f[x_i,\phi])\right) 
    $$
    - Let $f_{k}[x_i,\phi]$ denotes the k-th element in the output vector of $f[x_i,\phi]$, then:
    $$
    \hat{\phi} = \arg\min_{\phi} -\sum_{i=1}^I \left( f_{y_i}[x_i,\phi] - \log \left( \sum_{k'=1}^K \exp(f_{k'}[x_i,\phi]) \right) \right) 
    $$

### Multivariate Output
- Set Up:
    - Often, we wish to make more than one prediction with the same model, so the target output $y$ is a vector
    - For example, we might want to predict a molecule’s melting and boiling point at the same time (multi-variate regression)
    - Or the object class at every point in an image (multi-variate classification)
- Assumption:
    - We treat the probability distribution of each prediction type as independent
    - That is: $q(y|f[x,\phi]) = \prod_d q(y_d|f_d[x,\phi])$
    - Where $q(y|f[x,\phi])$ is the conditional joint distribution of $y$ predicted by the model
    - $q(y_d|f_d[x,\phi])$ is the conditional distribution of elemnt $y_d$ of the vector predicted by the model
    - $f_d[x,\phi]$ is the d-th set of outputs of the model, which describe the parameters of the conditional distribution over $y_d$
- Cross Entropy Loss:
$$
\begin{align*} 
\hat{\phi} &= \arg\min_{\phi} -\sum_{i=1}^I \log \left( \prod_d q(y_{i,d}|f_d[x_i,\phi]) \right) \\ 
&= \arg\min_{\phi} -\sum_{i=1}^I \sum_d \log \left( q(y_{i,d}|f_d[x_i,\phi]) \right) 
\end{align*}
$$